# Hello, Voyager

**EI@Home good first experiment #1 · Track A (continuous sequence modeling)**

In September 2016 the Green Bank Telescope pointed at Voyager 1, a human-made transmitter more than
20 billion km away, and recorded it with the same backend Breakthrough Listen uses to search
for technosignatures. Voyager is the perfect first signal for EI@Home: it's real, it's far away,
and we know exactly what it is. That makes it a **positive control**.

In this notebook we:

1. Download one Breakthrough Listen observation of Voyager 1.
2. Find its carrier and two telemetry sidebands with a 15-line de-Doppler search.
3. Cut short snippets: Voyager itself, plus real noise and synthetic drifting tones as controls.
4. Train a small GRU to **predict the next spectrogram frame**, on the controls only. Voyager is
   held out.
5. Compare the GRU with simple baselines.
6. Look inside: hidden states, and a linear probe that tries to read drift rate from them.

It runs in a few minutes on a laptop CPU. No GPU needed. For the bigger picture, see the
[README](../../README.md) and the [experiment protocol](../../docs/experiments.md).

## 0. Setup

From the repository root, run `pip install -r requirements.txt`, then open this notebook in
Jupyter. All the settings you might want to change are in the next cell.

In [ ]:
import os
import urllib.request
from pathlib import Path

import h5py
import hdf5plugin  # noqa: F401  (registers the bitshuffle filter used in Breakthrough Listen files)
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from torch import nn

SEED = 0
WINDOW = 64       # channels per snippet (about 180 Hz)
N_NOISE = 3000    # real-noise control snippets
N_SYNTH = 3000    # synthetic drifting-tone control snippets
MAX_DRIFT = 3.5   # largest synthetic drift rate, in channels per time step
HIDDEN = 64       # size of the GRU hidden state
EPOCHS = 40
BATCH = 128

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

## 1. Get the data

This is the Voyager 1 example observation from the Breakthrough Listen tutorials, the same file
used in the [turboSETI](https://github.com/UCBerkeleySETI/turbo_seti) and
[blimpy](https://github.com/UCBerkeleySETI/blimpy) READMEs. It holds one coarse channel at high
spectral resolution: about 2.8 Hz per channel and about 18 s per time step, which leaves only
**16 time steps** for the whole scan (see section 2.1 of the experiment protocol).

The file is downloaded into `data/` at the repository root (git ignores that folder). If you
already have a copy, set the `EIATHOME_VOYAGER_H5` environment variable to its path.

In [ ]:
DATA_URL = "http://blpd0.ssl.berkeley.edu/Voyager_data/Voyager1.single_coarse.fine_res.h5"


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "tracks").is_dir() and (path / "README.md").is_file():
            return path
    return start


ROOT = find_repo_root()
H5_PATH = Path(
    os.environ.get("EIATHOME_VOYAGER_H5", ROOT / "data" / "Voyager1.single_coarse.fine_res.h5")
)

if not H5_PATH.exists():
    H5_PATH.parent.mkdir(parents=True, exist_ok=True)
    partial = H5_PATH.with_suffix(".part")
    print(f"Downloading {DATA_URL}\n  -> {H5_PATH}")
    try:
        urllib.request.urlretrieve(DATA_URL, partial)
    except Exception as err:
        partial.unlink(missing_ok=True)
        raise RuntimeError(
            "Download failed. Fetch the file from the URL above in a browser and either save it "
            f"as {H5_PATH} or set EIATHOME_VOYAGER_H5 to wherever you saved it."
        ) from err
    partial.rename(H5_PATH)

print(f"Using {H5_PATH} ({H5_PATH.stat().st_size / 1e6:.0f} MB)")

In [ ]:
with h5py.File(H5_PATH, "r") as f:
    dset = f["data"]  # shape: (time, feed, frequency)
    header = dict(dset.attrs)
    data = dset[:, 0, :].astype(np.float32)


def attr(key):
    value = header[key]
    return value.decode() if isinstance(value, bytes) else value


fch1, foff, tsamp = float(attr("fch1")), float(attr("foff")), float(attr("tsamp"))  # MHz, MHz, s
n_t, n_chan = data.shape
foff_hz = foff * 1e6
freqs_mhz = fch1 + foff * np.arange(n_chan)


def drift_hz_per_s(channels_per_step):
    return channels_per_step * foff_hz / tsamp


print(f"Source:          {attr('source_name')}")
print(f"Shape:           {n_t} time steps x {n_chan:,} channels")
print(f"Channel width:   {foff_hz:.3f} Hz (negative: frequency falls as channel index rises)")
print(f"Time step:       {tsamp:.2f} s (scan length {n_t * tsamp / 60:.1f} min)")
print(f"Frequency range: {freqs_mhz.min():.4f} to {freqs_mhz.max():.4f} MHz")

## 2. Look at the whole coarse channel

Summing over time gives the spectrum of the full coarse channel. Breakthrough Listen
fine-resolution data typically shows two instrumental features before any real signal:

- a **DC spike**: a bright channel at the exact centre of the coarse channel, which is an artifact
  of the FFT and not a signal;
- **roll-off** at both edges of the coarse channel, from the polyphase filterbank.

We mask the centre channel and stay away from the edges. The printout tells you how bright the
centre channel actually is in this file.

In [ ]:
mid = n_chan // 2
spectrum = data.sum(axis=0)
neighbours = np.r_[spectrum[mid - 8:mid - 1], spectrum[mid + 2:mid + 9]]
print(f"Centre channel is {spectrum[mid] / np.median(neighbours):.1f}x the median of its neighbours")

block = 1024
n_block = n_chan // block
blocks = spectrum[: n_block * block].reshape(n_block, block)
block_freqs = freqs_mhz[: n_block * block].reshape(n_block, block).mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.semilogy(block_freqs, blocks.max(axis=1), lw=0.8, label="max of each 1,024 channels")
ax.semilogy(block_freqs, np.median(blocks, axis=1), lw=0.8, label="median of each 1,024 channels")
ax.axvline(freqs_mhz[mid], color="grey", ls=":", label="coarse-channel centre")
ax.set(xlabel="Frequency (MHz)", ylabel="Summed power (arbitrary units)",
       title="The full coarse channel, summed over the scan")
ax.legend(loc="upper right", fontsize=8)
plt.show()

EDGE = n_chan // 20  # ignore 5% of the band at each edge


def mask_dc_spike(arr, centre, half_width=1):
    # Replace the centre channels with the median of the channels around them.
    lo, hi = centre - half_width, centre + half_width + 1
    fill = np.median(np.concatenate([arr[:, lo - 8:lo], arr[:, hi:hi + 8]], axis=1), axis=1)
    arr[:, lo:hi] = fill[:, None]
    return arr


data = mask_dc_spike(data, mid)

## 3. Find the signals

First we flatten each frame: divide by the bandpass (estimated from medians of 1,024-channel
blocks) and convert to units of the noise level. In these units the noise has a standard
deviation of about 1.

A drifting signal moves across channels from frame to frame, so simply summing frames smears it
out. A **de-Doppler search** tries a range of drift rates, shifts each frame back by
`drift x time`, and sums. A real drifting tone lights up at its own drift rate. That's the idea
behind turboSETI, in about 15 lines.

The [turboSETI README](https://github.com/UCBerkeleySETI/turbo_seti) reports three hits in this
file: a carrier with SNR about 246 drifting at -0.373 Hz/s, and two telemetry sidebands about
22 kHz either side, with SNR about 31, drifting at -0.392 Hz/s. Let's see if we find them.

In the waterfalls below, channel index rises as frequency falls, so a negative drift rate moves
to the right.

In [ ]:
def flatten_frames(data, block=1024):
    n_block = data.shape[1] // block
    centres = (np.arange(n_block) + 0.5) * block
    channels = np.arange(data.shape[1])
    z = np.empty_like(data)
    for t, frame in enumerate(data):
        medians = np.median(frame[: n_block * block].reshape(n_block, block), axis=1)
        ratio = frame / np.interp(channels, centres, medians) - 1
        z[t] = ratio / (1.4826 * np.median(np.abs(ratio)))
    return z


def dedoppler(z, drifts):
    # For each channel, the best SNR over all trial drift rates, and that drift rate.
    steps = np.arange(z.shape[0])
    best_snr = np.full(z.shape[1], -np.inf, dtype=np.float32)
    best_drift = np.zeros(z.shape[1], dtype=np.float32)
    for drift in drifts:
        shifts = np.round(drift * steps).astype(int)
        total = sum(np.roll(z[t], -s) for t, s in enumerate(shifts))
        snr_at_drift = total / np.sqrt(z.shape[0])
        better = snr_at_drift > best_snr
        best_snr[better] = snr_at_drift[better]
        best_drift[better] = drift
    return best_snr, best_drift


def find_peaks(snr, n_max=3, threshold=15, exclusion=2 * WINDOW):
    snr = snr.copy()
    peaks = []
    while len(peaks) < n_max:
        idx = int(np.argmax(snr))
        if snr[idx] < threshold:
            break
        peaks.append(idx)
        snr[max(0, idx - exclusion): idx + exclusion] = -np.inf
    return peaks


z = flatten_frames(data)
snr, best_drift = dedoppler(z, drifts=np.arange(-4, 4.01, 0.25))  # channels per time step
snr[:EDGE] = -np.inf
snr[-EDGE:] = -np.inf
snr[mid - WINDOW: mid + WINDOW] = -np.inf  # in case the DC spike is wider than we masked
peaks = find_peaks(snr)
print(f"Found {len(peaks)} signals")

Next we measure each signal's drift rate directly: follow it through the frames, take the
power-weighted centre in each frame, and fit a straight line. We call the brightest signal the
carrier and the others sidebands.

In [ ]:
PUBLISHED_DRIFT = {"carrier": -0.373, "sideband": -0.392}  # Hz/s, from the turboSETI README


def track(z, start, drift, half=6):
    # Power-weighted centre of the signal in each frame, following the de-Doppler drift.
    positions = []
    for t in range(z.shape[0]):
        guess = int(round(start + drift * t))
        segment = np.clip(z[t, guess - half: guess + half + 1], 0, None)
        if segment.sum() > 0:
            positions.append(guess - half + (segment * np.arange(segment.size)).sum() / segment.sum())
        else:
            positions.append(guess)
    return np.array(positions)


signals = []
brightest = max(peaks, key=lambda p: snr[p])
for p in sorted(peaks, key=lambda p: freqs_mhz[p]):
    positions = track(z, p, best_drift[p])
    slope, intercept = np.polyfit(np.arange(n_t), positions, 1)
    signals.append({
        "role": "carrier" if p == brightest else "sideband",
        "start": p,
        "centre": intercept + slope * n_t / 2,  # channel position at mid-scan
        "drift": slope,                         # channels per time step
        "snr": float(snr[p]),
    })

print(f"{'role':<9} {'freq (MHz)':>12} {'SNR':>7} {'drift (Hz/s)':>13} {'turboSETI':>10}")
for s in signals:
    freq = fch1 + foff * s["centre"]
    print(f"{s['role']:<9} {freq:12.6f} {s['snr']:7.1f} {drift_hz_per_s(s['drift']):13.3f} "
          f"{PUBLISHED_DRIFT[s['role']]:10.3f}")

In [ ]:
fig, axes = plt.subplots(1, len(signals), figsize=(4 * len(signals), 3.5), squeeze=False)
for ax, s in zip(axes[0], signals):
    lo = int(round(s["centre"])) - WINDOW // 2
    ax.imshow(np.arcsinh(z[:, lo:lo + WINDOW]), aspect="auto", origin="lower", cmap="viridis",
              extent=[0, WINDOW, 0, n_t * tsamp])
    ax.set(title=f"{s['role']} ({drift_hz_per_s(s['drift']):.3f} Hz/s)",
           xlabel="Channel in window", ylabel="Time (s)")
fig.suptitle("Voyager 1 waterfalls (colour: asinh of power in noise units)")
plt.tight_layout()
plt.show()

## 4. Build snippets

A **snippet** is 16 time steps x 64 channels. We make three groups:

| Group | How it's made | Used for |
|---|---|---|
| `noise` | Windows of this observation with no detected signal | Training (a real-noise control) |
| `synthetic` | Noise windows with an injected drifting tone of known drift rate and brightness | Training (control with known answers) |
| `voyager` | Windows around the carrier and sidebands, shifted by a few channels | **Held out.** Never used for training |

Noise and synthetic snippets are split by frequency into train, validation and test regions, so
that test snippets come from a different part of the band (protocol section 2.4).

Each snippet is normalized on its own (median and robust scale), then passed through `asinh` to
compress the huge dynamic range: Voyager's carrier is far brighter than the noise.

The synthetic tones are deliberately simple: a narrow line moving at a constant rate, smeared by
its motion during each time step. If Voyager doesn't look like them to the model, the line shape
is the first thing to suspect.

In [ ]:
def normalise(snippet):
    med = np.median(snippet)
    scale = 1.4826 * np.median(np.abs(snippet - med))
    return np.arcsinh((snippet - med) / scale).astype(np.float32)


def inject_tone(window, drift, flux, centre, width=0.6, n_sub=16):
    # Add a tone moving at `drift` channels per step; `flux` is its total power per frame.
    out = window.copy()
    channels = np.arange(window.shape[1])
    for t in range(window.shape[0]):
        positions = centre + drift * (t + np.arange(n_sub) / n_sub - n_t / 2)
        profile = np.exp(-0.5 * ((channels[None, :] - positions[:, None]) / width) ** 2).sum(axis=0)
        out[t] += flux * profile / (n_sub * width * np.sqrt(2 * np.pi))
    return out


# Channels to keep clear of: every signal's track, plus the coarse-channel centre.
busy = [(s["start"] - 2 * WINDOW, s["start"] + s["drift"] * n_t + 2 * WINDOW) for s in signals]
busy.append((mid - WINDOW, mid + WINDOW))


def is_clear(start):
    end = start + WINDOW
    return all(end < min(a, b) or start > max(a, b) for a, b in busy)


# Split the usable band into train / validation / test regions with a gap between them.
usable = np.array([EDGE, n_chan - EDGE - WINDOW])
cuts = usable[0] + (usable[1] - usable[0]) * np.array([0, 0.70, 0.85, 1.0])
REGIONS = {"train": (cuts[0], cuts[1] - WINDOW), "val": (cuts[1], cuts[2] - WINDOW),
           "test": (cuts[2], cuts[3])}
FRACTION = {"train": 0.70, "val": 0.15, "test": 0.15}


def random_windows(n, region):
    lo, hi = REGIONS[region]
    starts = []
    while len(starts) < n:
        start = int(rng.integers(lo, hi))
        if is_clear(start):
            starts.append(start)
    return starts


X, group, split, drift, flux = [], [], [], [], []

for region, frac in FRACTION.items():
    for start in random_windows(int(N_NOISE * frac), region):
        X.append(normalise(z[:, start:start + WINDOW]))
        group.append("noise"); split.append(region); drift.append(np.nan); flux.append(0.0)

    for start in random_windows(int(N_SYNTH * frac), region):
        d = rng.uniform(-MAX_DRIFT, MAX_DRIFT)
        f = np.exp(rng.uniform(np.log(5), np.log(1000)))
        centre = WINDOW / 2 + rng.uniform(-4, 4)
        X.append(normalise(inject_tone(z[:, start:start + WINDOW], d, f, centre)))
        group.append("synthetic"); split.append(region); drift.append(d); flux.append(f)

for s in signals:
    for offset in range(-4, 5):
        start = int(round(s["centre"])) - WINDOW // 2 + offset
        X.append(normalise(z[:, start:start + WINDOW]))
        group.append(s["role"]); split.append("heldout"); drift.append(s["drift"]); flux.append(np.nan)

X = np.stack(X)
group, split = np.array(group), np.array(split)
drift, flux = np.array(drift), np.array(flux)

for name in ["noise", "synthetic", "carrier", "sideband"]:
    counts = {r: int(((group == name) & (split == r)).sum()) for r in ["train", "val", "test", "heldout"]}
    print(f"{name:<10}", counts)

## 5. Train a GRU to predict the next frame

The GRU reads frames `0..t` and predicts frame `t+1`. It has about 30,000 parameters and trains
only on the noise and synthetic snippets.

To know whether it learned anything, we compare it with three baselines:

- **Mean:** always predict the average training frame. This is the error you get by knowing nothing.
- **Persistence:** predict that the next frame equals the current one.
- **Linear:** one linear map from the current frame to the next, fit by least squares. A tone at a
  *fixed* drift rate is just a shift, which is linear, but the drift rate changes from snippet to
  snippet, so a single linear map can't follow them all. The GRU can, by working out the drift
  from earlier frames.

**Prediction gain** = 1 - MSE(model) / MSE(mean baseline). Zero means no better than knowing
nothing. We score predictions of frames 2 to 15, so each prediction has at least two frames of
history to work with.

In [ ]:
class NextFrameGRU(nn.Module):
    def __init__(self, n_chan=WINDOW, hidden=HIDDEN):
        super().__init__()
        self.gru = nn.GRU(n_chan, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_chan)

    def forward(self, x):                 # x: (batch, time, channels)
        states, _ = self.gru(x)           # states: (batch, time, hidden)
        return self.head(states), states  # output at step t predicts frame t + 1


train_mask = (split == "train") & np.isin(group, ["noise", "synthetic"])
val_mask = (split == "val") & np.isin(group, ["noise", "synthetic"])
X_train, X_val = torch.tensor(X[train_mask]), torch.tensor(X[val_mask])

model = NextFrameGRU()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
optimiser = torch.optim.Adam(model.parameters(), lr=3e-3)


def next_frame_loss(x):
    prediction, _ = model(x[:, :-1])
    return ((prediction - x[:, 1:]) ** 2).mean()


history, best = [], (np.inf, None)
for epoch in range(EPOCHS):
    model.train()
    order = torch.randperm(len(X_train))
    for i in range(0, len(order), BATCH):
        loss = next_frame_loss(X_train[order[i:i + BATCH]])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    model.eval()
    with torch.no_grad():
        val_loss = next_frame_loss(X_val).item()
    history.append(val_loss)
    if val_loss < best[0]:
        best = (val_loss, {k: v.clone() for k, v in model.state_dict().items()})

model.load_state_dict(best[1])
plt.figure(figsize=(5, 3))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.title("Training curve")
plt.show()

In [ ]:
def gru_predict(x):
    with torch.no_grad():
        prediction, _ = model(torch.tensor(x[:, :-1]))
    return prediction.numpy()


# Baselines, fit on the training snippets only.
mean_frame = X[train_mask][:, 1:].mean(axis=(0, 1))
inputs = X[train_mask][:, :-1].reshape(-1, WINDOW)
targets = X[train_mask][:, 1:].reshape(-1, WINDOW)
design = np.c_[inputs, np.ones(len(inputs))]
linear_map = np.linalg.solve(design.T @ design + 1e-3 * np.eye(WINDOW + 1), design.T @ targets)

PREDICTORS = {
    "mean": lambda x: np.broadcast_to(mean_frame, x[:, 1:].shape),
    "persistence": lambda x: x[:, :-1],
    "linear": lambda x: np.c_[x[:, :-1].reshape(-1, WINDOW),
                              np.ones(len(x) * (n_t - 1))].dot(linear_map).reshape(x[:, 1:].shape),
    "GRU": gru_predict,
}


def mse(x, predictor):
    # Mean squared error on frames 2..15 (each prediction has at least two frames of history).
    return float(((predictor(x)[:, 1:] - x[:, 2:]) ** 2).mean())


EVAL_SETS = {
    "noise (test)": (group == "noise") & (split == "test"),
    "synthetic (test)": (group == "synthetic") & (split == "test"),
    "Voyager carrier": group == "carrier",
    "Voyager sidebands": group == "sideband",
}

print(f"{'prediction gain':<20}" + "".join(f"{name:>13}" for name in PREDICTORS))
for label, mask in EVAL_SETS.items():
    if not mask.any():
        continue
    floor = mse(X[mask], PREDICTORS["mean"])
    gains = [1 - mse(X[mask], p) / floor for p in PREDICTORS.values()]
    print(f"{label:<20}" + "".join(f"{g:13.3f}" for g in gains))

In [ ]:
# What the predictions look like for the carrier (unshifted window).
example = X[(group == "carrier")][4:5]
panels = {"actual": example[0, 2:], "GRU": gru_predict(example)[0, 1:],
          "persistence": example[0, 1:-1], "linear": PREDICTORS["linear"](example)[0, 1:]}

fig, axes = plt.subplots(1, len(panels), figsize=(14, 3.2), sharey=True)
for ax, (name, img) in zip(axes, panels.items()):
    ax.imshow(img, aspect="auto", origin="lower", cmap="viridis", vmin=-1, vmax=example.max(),
              extent=[0, WINDOW, 1.5, n_t - 0.5])
    ax.set(title=name, xlabel="Channel in window")
axes[0].set_ylabel("Frame")
fig.suptitle("Voyager carrier: next-frame predictions (the model never trained on Voyager)")
plt.tight_layout()
plt.show()

## 6. Look inside the GRU

To predict where a drifting tone goes next, the GRU has to keep track of how fast it's moving.
If it does, the drift rate should be readable from its **hidden state**.

We summarize each snippet by the GRU's hidden state after the last frame, and then:

1. project the hidden states to 2D with PCA (fit on training snippets) and see where Voyager lands
   relative to the controls;
2. fit a **linear probe** (ridge regression) that predicts drift rate from the hidden state, using
   synthetic training snippets only, and then ask it about Voyager.

In [ ]:
def final_state(x):
    with torch.no_grad():
        _, states = model(torch.tensor(x))
    return states[:, -1].numpy()


H = final_state(X)
pca = PCA(n_components=2).fit(H[train_mask])
P = pca.transform(H)

fig, ax = plt.subplots(figsize=(7, 5.5))
test = split == "test"
ax.scatter(*P[test & (group == "noise")].T, s=6, c="lightgrey", label="noise (test)")
synthetic_test = test & (group == "synthetic")
points = ax.scatter(*P[synthetic_test].T, s=8, c=drift_hz_per_s(drift[synthetic_test]),
                    cmap="coolwarm", label="synthetic (test)")
ax.scatter(*P[group == "carrier"].T, s=160, marker="*", c="black", label="Voyager carrier")
ax.scatter(*P[group == "sideband"].T, s=60, marker="^", c="black", label="Voyager sidebands")
fig.colorbar(points, label="Synthetic drift rate (Hz/s)")
ax.set(xlabel="PC 1", ylabel="PC 2", title="GRU hidden states after the last frame")
ax.legend(loc="best", fontsize=8)
plt.show()

In [ ]:
synthetic_train = train_mask & (group == "synthetic")
probe = RidgeCV(alphas=np.logspace(-3, 3, 13)).fit(H[synthetic_train], drift[synthetic_train])

true_hz = drift_hz_per_s(drift[synthetic_test])
predicted_hz = drift_hz_per_s(probe.predict(H[synthetic_test]))
print(f"Probe on synthetic test snippets: R^2 = {r2_score(true_hz, predicted_hz):.3f}, "
      f"mean absolute error = {mean_absolute_error(true_hz, predicted_hz):.3f} Hz/s\n")

print(f"{'Voyager':<9} {'probe (Hz/s)':>18} {'measured (Hz/s)':>16} {'turboSETI':>10}")
for role in ["carrier", "sideband"]:
    mask = group == role
    readout = drift_hz_per_s(probe.predict(H[mask]))
    measured = drift_hz_per_s(drift[mask]).mean()
    print(f"{role:<9} {readout.mean():10.3f} ± {readout.std():.3f} {measured:16.3f} "
          f"{PUBLISHED_DRIFT[role]:10.3f}")

fig, ax = plt.subplots(figsize=(6.5, 5))
points = ax.scatter(true_hz, predicted_hz, s=6, c=np.log10(flux[synthetic_test]), cmap="viridis")
fig.colorbar(points, label="Synthetic brightness (log10 of flux per frame)")
for role, marker, size in [("carrier", "*", 160), ("sideband", "^", 60)]:
    mask = group == role
    ax.scatter(drift_hz_per_s(drift[mask]), drift_hz_per_s(probe.predict(H[mask])),
               marker=marker, s=size, c="black", label=f"Voyager {role}")
limit = drift_hz_per_s(MAX_DRIFT)
ax.plot([-abs(limit), abs(limit)], [-abs(limit), abs(limit)], "k:", lw=1)
ax.set(xlabel="True drift rate (Hz/s)", ylabel="Probe readout (Hz/s)",
       title="Linear probe: drift rate from hidden state")
ax.legend(fontsize=8)
plt.show()

### Does the order of frames matter?

If the GRU really uses the signal's dynamics, scrambling the order of the frames should destroy
its advantage. We shuffle the time order of every snippet (the same shuffle for all of them) and
score again.

In [ ]:
shuffle = rng.permutation(n_t)
print(f"{'GRU prediction gain':<20} {'original':>10} {'frames shuffled':>16}")
for label, mask in EVAL_SETS.items():
    if not mask.any():
        continue
    original, shuffled = X[mask], X[mask][:, shuffle]
    gain = 1 - mse(original, gru_predict) / mse(original, PREDICTORS["mean"])
    gain_shuffled = 1 - mse(shuffled, gru_predict) / mse(shuffled, PREDICTORS["mean"])
    print(f"{label:<20} {gain:10.3f} {gain_shuffled:16.3f}")

## 7. How to read the results

| Result | A healthy outcome | If it isn't |
|---|---|---|
| Noise, prediction gain | Close to 0 for every model: noise is unpredictable | Something in preprocessing creates structure. Check normalization (protocol section 6) |
| Synthetic, GRU vs. baselines | GRU gain above persistence and linear | The GRU didn't learn to track drift. Train longer, or check the injection |
| Voyager, GRU gain | Positive, even though the GRU never saw Voyager | Real signals differ from the synthetic tones. Compare their line shapes |
| Hidden-state plot | Voyager among synthetic tones of similar drift, away from noise | Same as above: the controls don't cover what Voyager looks like |
| Linear probe | High R² on synthetic tests, and a Voyager readout near the measured drift | The hidden state doesn't encode drift in a linear way, or doesn't encode it at all |
| Shuffled frames | GRU gain drops sharply | The model relies on per-frame brightness rather than dynamics |

**Caveats, so we stay honest:**

- **Only 16 time steps.** That's all this data product offers per scan (protocol section 2.1).
  It's enough for drift, but not for modulation, which needs raw voltages.
- **Voyager is a positive control, not a candidate.** We know exactly what it is. The point is to
  check that the pipeline finds what we know is there.
- **The Voyager snippets aren't independent.** They're shifted views of three signals in one
  observation.
- **One observation, one model, one seed.** A real result needs the replication and criteria in
  protocol section 7.

## 8. Next steps (pick one!)

1. **Flip it.** Add Voyager snippets, or turboSETI hits from another observation, to the
   *training* set and see how the hidden states change. That's the EI@Home premise: candidates
   as training data.
2. **GRU vs. tiny transformer** at the same parameter count (good first experiment #3).
3. **De-drift** the snippets before training. Is there anything left to predict?
4. **Swap the hand-made injection for [setigen](https://github.com/bbrzycki/setigen)**, with more
   realistic line shapes.
5. **Swap the GRU** for minGRU or mamba.py.
6. **Write it up** with the reporting template in protocol section 8, and share it in an issue.